# 03 — Design and evaluate chunk boundaries

**Track:** Beginner · **Stage:** Ingestion & Chunking

Chunking is not clerical preprocessing. Chunking decides what facts are visible together. A bad boundary can make the right answer unretrievable even when the corpus contains it. In this lab, you will compare standard splitters using **LangChain** and measure how chunk boundaries affect retrieval context.

## What you will build

- A comparison between character-based splitting and recursive semantic splitting.
- An evaluation of how chunk size and overlap affect context dilution.

## Setup: LangChain Splitters

We will use LangChain's built-in text splitters. We don't need an LLM for this step, just the parsing utilities.

In [ ]:
# !pip install langchain-text-splitters

from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def print_chunks(chunks):
    for i, chunk in enumerate(chunks):
        print(f"--- Chunk {i+1} [Length: {len(chunk.page_content)}] ---")
        print(chunk.page_content)
        print()

## 1. The Problem with Naive Chunking

Let's define a document where critical context spans multiple sentences. If we split blindly by character count, we might separate a metric from its subject.

In [ ]:
raw_text = """# Q2 2025 Financial Review

## Cloud Infrastructure
The migration to the new distributed cluster architecture was completed in May. 
As a direct result of these redundant systems and cross-region backups, 
costs increased by 14% compared to the previous quarter. 
However, uptime improved to 99.999%.
"""

doc = Document(page_content=raw_text, metadata={"source": "q2_review.md"})

# Naive character splitting (forces a hard split at exactly 100 characters)
naive_splitter = CharacterTextSplitter(
    separator="",
    chunk_size=100,
    chunk_overlap=0
)

naive_chunks = naive_splitter.split_documents([doc])
print("Naive Character Splitting:\n")
print_chunks(naive_chunks)

Notice how Chunk 3 or 4 might contain the phrase "costs increased by 14%" but has completely lost the context of *what* increased (Cloud Infrastructure). If a user asks "What increased by 14%?", the retriever might find the chunk with the number, but the LLM won't know the subject.

## 2. Recursive Character Splitting (The Industry Standard)

A `RecursiveCharacterTextSplitter` tries to split on paragraphs (`\n\n`), then sentences (`.`), then words (` `), keeping semantic units intact as much as possible while respecting the chunk size.

In [ ]:
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=20,
    separators=["\n\n", "\n", " ", ""]
)

recursive_chunks = recursive_splitter.split_documents([doc])
print("Recursive Semantic Splitting:\n")
print_chunks(recursive_chunks)

## 3. Metadata Preservation

When you chunk a document, the resulting chunks must inherit the parent document's metadata. This is critical for filtering and citations.

In [ ]:
for chunk in recursive_chunks:
    print(f"Chunk Metadata: {chunk.metadata}")

## Reflection

1. **Overlap:** Why did we set `chunk_overlap=20`? Overlap ensures that concepts split across a boundary still share context.
2. **Advanced Chunking:** For complex documents, consider `MarkdownHeaderTextSplitter` which splits explicitly on `#`, `##`, and `###` headers, attaching the header path to the metadata. This is highly recommended for structured enterprise wikis.